# Test 배치91 이탈점수 검증 — 사전작업

이탈점수 로직이 완성되기 전에 먼저 준비할 수 있는 작업들을 처리한다.
- Batch91 공정진행률 계산
- 실제 Fault 시작 지점(진행률 7.8%) 확인
- Test셋 정상 배치 19개 리스트 정리
- 이탈점수 함수만 끼워넣으면 바로 돌아가는 스켈레톤 코드 준비

이탈점수 함수가 완성되는 즉시 이어서 검증에 들어간다.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('merged_data_clean.csv')
batch_col = '배치번호'
time_col = '발효시간'

# Batch91(Test셋 유일 Fault 배치) 데이터만 추출
batch91 = df[df[batch_col] == 91].sort_values(time_col).copy()

# 공정진행률 계산 (골든배치 만들 때와 동일한 방식: 발효시간 ÷ 실제 최종시간)
# 91번은 이미 끝난 배치라 최종시간을 알고 있으므로 미래정보 누수 문제 없음
batch91['공정진행률'] = batch91[time_col] / batch91[time_col].max()

print(f"Batch91 데이터 shape: {batch91.shape}")
print(f"발효시간 범위: {batch91[time_col].min()} ~ {batch91[time_col].max()}")
print(batch91[[time_col, '공정진행률']].head())

Batch91 데이터 shape: (1290, 40)
발효시간 범위: 0.2 ~ 258.0
        발효시간     공정진행률
102410   0.2  0.000775
102411   0.4  0.001550
102412   0.6  0.002326
102413   0.8  0.003101
102414   1.0  0.003876


### 0. Batch91 데이터 확인 결과

- 총 1,290개 행, 40개 컬럼 — 12분(0.2h) 간격으로 258시간 동안 측정된 배치
- 발효시간 0.2h부터 시작하며, 공정진행률이 0에 가깝게 정상적으로 계산됨(0.2h ÷ 258h ≈ 0.000775)
- 이 배치는 이미 완료된 Test셋 배치라 최종시간(258h)을 알고 있는 상태 → 공정진행률 계산에 미래정보 누수 문제 없음(사후분석 용도)

다음: 실제 Fault 시작 지점이 이전에 확인했던 진행률 7.8%와 일치하는지 재확인한다.

In [2]:
# Fault_ref(결함라벨_구간) 컬럼으로 실제 Fault가 언제 시작되는지 확인
# 이전에 정리한 내용 기준 배치91의 Fault 시작 진행률은 7.8%였는데, 실제 데이터로 재확인

fault_start = batch91[batch91['결함라벨_구간'] == 1]

if len(fault_start) > 0:
    first_fault_row = fault_start.iloc[0]
    print(f"Fault 시작 시점 (발효시간): {first_fault_row[time_col]}h")
    print(f"Fault 시작 시점 (공정진행률): {first_fault_row['공정진행률']*100:.2f}%")
else:
    print("이 배치엔 결함라벨_구간=1인 행이 없습니다 (컬럼명/배치 확인 필요)")

# 참고용: 결함라벨_배치(넓은 정의)로도 확인 - 91~100 전체가 1이어야 정상
print(f"\n결함라벨_배치 값 확인 (전체 배치 1이어야 정상): {batch91['결함라벨_배치'].unique()}")

Fault 시작 시점 (발효시간): 20.0h
Fault 시작 시점 (공정진행률): 7.75%

결함라벨_배치 값 확인 (전체 배치 1이어야 정상): [1]


### 1. Fault 시작 지점 확인 결과

- 실제 Fault 시작 시점: 발효시간 20.0h, 공정진행률 7.75%
- 이전에 확인했던 "진행률 7.8%"와 거의 일치함(반올림 차이 수준) — 재확인 완료
- 결함라벨_배치 값이 전체 1로 나와, 91번이 Fault 그룹(91~100)에 정상적으로 속해있음도 확인됨

이제 이 7.75% 지점을 기준선(세로 점선)으로 삼아, 이탈점수 그래프에서 실제 Fault 시작과 이탈점수 급등 시점이 일치하는지 나중에 비교한다.

다음: Test셋 정상 배치 19개 리스트를 정리한다.

In [3]:
# Test셋 = GroupShuffleSplit(test_size=0.2, random_state=42)로 이미 분할된 배치들
# 이전 EDA에서 확인한 Test 구성: RC6·OC7·APC6·Fault1(=91번), 총 20개
# 여기서는 91번(Fault)을 제외한 "정상 라벨" 19개만 별도로 정리

# 주의: 실제 Test 배치 번호 리스트는 GroupShuffleSplit 실행 시 저장해뒀던 값을 그대로 써야 함
# (아래는 자리표시용 — 기존 노트북에서 쓴 test_batches 변수가 있다면 그걸 불러오는 게 정확함)

# 방법 A: 기존 노트북에서 저장해둔 test_batches 리스트를 그대로 사용 (권장)
# test_batches = [...]  # 기존 03_eda.ipynb 또는 골든배치 노트북에서 복사

# 방법 B: 여기서 다시 분할해야 한다면 (기존과 동일한 random_state=42 사용 필수)
from sklearn.model_selection import GroupShuffleSplit

strategy_col = '제어전략그룹'
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
batch_summary = df.groupby(batch_col)[strategy_col].first().reset_index()

train_idx, test_idx = next(gss.split(batch_summary, groups=batch_summary[batch_col]))
test_batches_all = batch_summary.iloc[test_idx][batch_col].tolist()

test_batches_normal = [b for b in test_batches_all if b != 91]

print(f"Test셋 전체: {len(test_batches_all)}개")
print(f"Test셋 정상 배치(91 제외): {len(test_batches_normal)}개")
print(test_batches_normal)

Test셋 전체: 20개
Test셋 정상 배치(91 제외): 19개
[1, 5, 11, 13, 19, 23, 31, 32, 34, 40, 45, 46, 54, 71, 74, 77, 78, 81, 84]


### 2. Test셋 정상 배치 19개 정리 결과

- Test셋 전체 20개 중 91번(Fault)을 제외한 19개를 정리함
- 배치 목록: [1, 5, 11, 13, 19, 23, 31, 32, 34, 40, 45, 46, 54, 71, 74, 77, 78, 81, 84]
- GroupShuffleSplit(random_state=42)로 재분할했으며, 기존 EDA에서 사용한 분할과 동일한 시드를 사용해 일관성 유지

이 19개는 이탈점수 로직 완성 후, 91번과 점수 분포를 비교하는 대조군으로 사용한다
(단일 Fault 배치 결과를 전체 성능으로 일반화하지 않기 위함).

다음: 이탈점수 함수만 끼워넣으면 바로 돌아가는 스켈레톤 코드를 준비한다.

In [7]:
# ⚠️ 이 함수는 아직 미완성 — 팀원1의 이탈점수 로직이 완성되면 안쪽 계산 부분만 교체
# 지금은 골든배치 프로파일(bands)과 std_floor를 입력받아,
# 특정 배치의 이탈점수 시계열을 반환하는 '틀'만 미리 짜둔 것
# (수정: Fault 배치는 bands에 없으므로 compare_strategy로 비교 대상을 직접 지정받도록 변경)

def calc_deviation_score(batch_id, bands, feature_cols, std_floor=None, compare_strategy=None):
    """
    batch_id: 이탈점수를 계산할 배치 번호
    bands: 전략별 골든배치 기준 프로파일 딕셔너리 (build_golden_band 결과)
    feature_cols: z-score 계산에 쓸 변수 목록 (12개)
    std_floor: 변수별 최소 표준편차 하한값 (팀원1 작업 완료 후 연결)
    compare_strategy: Fault 배치처럼 자체 골든배치 프로파일이 없는 경우,
                       어느 전략(RC/OC/APC) 프로파일과 비교할지 명시 (예: 'RC')

    반환: 공정진행률별 종합 이탈점수(pd.Series)
    """
    sub = df[df[batch_col] == batch_id].sort_values(time_col)
    progress = sub[time_col] / sub[time_col].max()

    strategy = df[df[batch_col] == batch_id][strategy_col].iloc[0]

    # Fault 배치는 bands에 없으므로, 비교할 기준 전략을 명시적으로 지정받음
    if strategy == 'Fault':
        if compare_strategy is None:
            raise ValueError(f"배치{batch_id}는 Fault 그룹입니다. compare_strategy를 지정하세요 (예: 'RC', 'OC', 'APC')")
        band = bands[compare_strategy]
    else:
        band = bands[strategy]

    # TODO: 이탈점수 함수 완성되면 아래 부분 교체
    # z_scores = calc_z_score(sub[feature_cols], band, std_floor)
    # deviation_score = z_scores.mean(axis=1)

    deviation_score = None  # placeholder

    return progress, deviation_score

print("스켈레톤 함수 수정 완료 — Fault 배치는 compare_strategy 지정 필요, 팀원1 로직 완성되는 대로 TODO 부분만 교체하면 됩니다.")

스켈레톤 함수 수정 완료 — Fault 배치는 compare_strategy 지정 필요, 팀원1 로직 완성되는 대로 TODO 부분만 교체하면 됩니다.


In [5]:
# 팀원1 로직 완성되면, 이 반복문 하나로 91번+정상19개 전부 한 번에 이탈점수 계산 가능
# 지금은 결과를 담을 그릇(딕셔너리)만 미리 만들어둠

target_batches = [91] + test_batches_normal  # Fault 1개 + 정상 19개 = 총 20개

deviation_results = {}  # {배치번호: (progress, deviation_score)} 저장할 그릇

for b in target_batches:
    # TODO: 팀원1 로직 완성되면 아래 주석 해제
    # progress, score = calc_deviation_score(b, bands, feature_cols, std_floor)
    # deviation_results[b] = (progress, score)
    pass

print(f"대상 배치 {len(target_batches)}개 (Fault 1 + 정상 {len(test_batches_normal)}) 준비 완료")
print("TODO 주석만 해제하면 바로 전체 계산 시작 가능")

대상 배치 20개 (Fault 1 + 정상 19) 준비 완료
TODO 주석만 해제하면 바로 전체 계산 시작 가능
